In [1]:
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import Dense, Dropout

from tensorflow.keras.optimizers import Adam

In [6]:
DATA_PATH = r"C:/Users/Dell/Downloads/caltech-101/caltech-101/101_ObjectCategories/101_ObjectCategories"

categories = [
    "airplanes",
    "butterfly",
    "wheelchair",
    "sunflower",
    "starfish"
]

print("Selected Categories:")
print(categories)

Selected Categories:
['airplanes', 'butterfly', 'wheelchair', 'sunflower', 'starfish']


In [7]:
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    DATA_PATH,
    target_size=(224, 224),
    classes=categories,
    class_mode="categorical",
    batch_size=16,
    subset="training"
)

test_data = datagen.flow_from_directory(
    DATA_PATH,
    target_size=(224, 224),
    classes=categories,
    class_mode="categorical",
    batch_size=16,
    subset="validation",
    shuffle=False
)

print("Training images:", train_data.samples)
print("Testing images:", test_data.samples)

Found 898 images belonging to 5 classes.
Found 223 images belonging to 5 classes.
Training images: 898
Testing images: 223


In [8]:
vgg16 = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

print("VGG16 loaded successfully")

VGG16 loaded successfully


In [9]:
vgg16.trainable = False
print("VGG16 convolutional layers frozen")

VGG16 convolutional layers frozen


In [10]:
model = Sequential([
    vgg16,

    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dropout(0.5),

    Dense(5, activation="softmax")
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)                   │ (None, 7, 7, 512)           │      14,714,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 512)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │          65,664 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 5)                   │             325 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 14,788,933 (56.42 MB)

 Trainable params: 74,245 (290.02 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [11]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully")

Model compiled successfully


In [ ]:
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=5
)

Epoch 1/5
57/57 ━━━━━━━━━━━━━━━━━━━━ 374s 6s/step - accuracy: 0.5312 - loss: 3.2950 - val_accuracy: 0.8430 - val_loss: 0.7704
Epoch 2/5
23/57 ━━━━━━━━━━━━━━━━━━━━ 2:58 5s/step - accuracy: 0.7288 - loss: 1.6444

In [ ]:
loss, accuracy = model.evaluate(test_data)

print("Test Accuracy:", accuracy)
print("Test Accuracy (%):", accuracy * 100)

In [ ]:
test_data.reset()

images, labels = next(test_data)

predictions = model.predict(images, verbose=0)

class_names = list(test_data.class_indices.keys())

for i in range(5):

    actual = class_names[np.argmax(labels[i])]
    predicted = class_names[np.argmax(predictions[i])]

    image = images[i]

    # Convert image back to displayable range
    image = image - image.min()
    image = image / image.max()

    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.axis("off")

    plt.title(
        "Actual: " + actual +
        "\nPredicted: " + predicted
    )

    plt.show()